## How to Process IoT Device JSON Data Using Dataset

####Reading JSON as a Dataset

Use the Scala case class *DeviceIoTData* to convert the JSON device data into a dataframe. There is GeoIP information for each device entry:
* IP address
* ISO-3166-1 two and three letter codes
* Country Name
* Latitude and longitude

With these attributes as part of the device data, we can map and visualize them as needed. For each IP associated with a *device_id*, I optained the above attributes from a webservice at http://freegeoip.net/csv/ip

*{"device_id": 198164, "device_name": "sensor-pad-198164owomcJZ", "ip": "80.55.20.25", "cca2": "PL", "cca3": "POL", "cn": "Poland", "latitude": 53.080000, "longitude": 18.620000, "scale": "Celsius", "temp": 21, "humidity": 65, "battery_level": 8, "c02_level": 1408, "lcd": "red", "timestamp" :1458081226051 }*

This dataset is avaialbe from Public S3 bucket //databricks-public-datasets/data/iot or https://github.com/dmatrix/examples/blob/master/spark/databricks/notebooks/py/data/iot_devices.json


In [0]:
%fs ls "s3://databricks-public-datasets/data/iot/"

In [0]:
# read the json file and create the dataframe
import pandas as pd

file_location = "https://raw.githubusercontent.com/dmatrix/examples/master/spark/databricks/notebooks/py/data/iot_devices.json"

pdf = pd.read_json(file_location, lines=True)
df = spark.createDataFrame(pdf)

display(df)



In [0]:
# Create a view or table

temp_table_name = "iot_devices_json"

df.createOrReplaceTempView(temp_table_name)

In [0]:
df.count()

Displaying your Dataset

In [0]:
display(df)

#### Data exploration

Top five entries

In [0]:
df.take(5)


For all relational expressions, the [Catalyst Optimizer](https://databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html) will formulate an optimized logical and physical plan for execution, and [Tungsten](https://databricks.com/blog/2015/04/28/project-tungsten-bringing-spark-closer-to-bare-metal.html) engine will optimize the generated code. For our *DeviceIoTData*, it will use its standard encoders to optimize its binary internal representation, hence decrease the size of generated code, minimize the bytes transfered over the networks between nodes, and execute faster.

For instance, let's first filter the device dataset on *temp* and *humidity* attributes with a predicate and display the first 10 items.



In [0]:
# issue select, map, filter operations on the dataframes

from pyspark.sql.functions import col, asc
TempFilter = df.filter(col("temp") > 30).filter(col("humidity") > 70)
display(TempFilter)


Use filter to filter out dataframe rows that met the temperature and humidity predicate

In [0]:
# filter out rows that meet the temperature and humimdity predicate
TempFilter10 = df.filter(col("temp") > 30).filter(col("humidity") > 70).take(10)
TempFilter10

In [0]:
# Mapping four fields- temp, device_name, device_id, cca3 
dfTempMap = df.where((col("temp") > 25)).select("temp", "device_name", "device_id", "cca3")
dfTempMap

In [0]:
display(dfTempMap)

Now use the filter() method that is equivalent as the where() method used above.

In [0]:
dfTemp25 = df.filter(col("temp") > 25).select("temp", "device_name", "device_id", "cca3")

display(dfTemp25)

select() where battery_level is greater than 6, sort in asceding order on C02_level.

In [0]:
display(df.select("battery_level", "c02_level", "device_name").where(col("battery_level") > 6).sort(col("c02_level")))

Let's see how to use groupBy() and avg(). 
Let's take all temperatures readings > 25, along with their corresponding devices' humidity, groupBy ccca3 country code, and compute averages. Plot the resulting Dataset.

In [0]:
from pyspark.sql.functions import avg

dfAvgTmp = df.filter(col("temp") > 25).select("temp", "humidity", "cca3").groupBy("cca3").agg(avg("temp"), avg("humidity"))


display(dfAvgTmp)


#### Visualizing datasets

**Finally, the fun bit!**

Data without visualization without a narrative arc, to infer insights or to see a trend, is useless. We always desire to make sense of the results.

By saving our Dataset, as a temporary table, I can issue complex SQL queries against it and visualize the results, using notebook's myriad plotting options.

In [0]:
df.createOrReplaceTempView("iot_device_data")

Count all devices for a partiular country and map them

In [0]:
%sql select cca3, count(distinct device_id) as device_id from iot_device_data group by cca3 order by device_id desc limit 100

Let's visualize the results as a pie chart and distribution for devices in the country where C02 are high.

In [0]:
%sql select cca3, c02_level from iot_device_data where c02_level > 1400 order by c02_level desc

Select all countries' devices with high-levels of C02 and group by cca3 and order by device_ids 

In [0]:
%sql select cca3, count(distinct device_id) as device_id from iot_device_data where lcd == 'red' group by cca3 order by device_id desc limit 100

find out all devices in countries whose batteries need replacements 

In [0]:
%sql select cca3, count(distinct device_id) as device_id from iot_device_data where battery_level == 0 group by cca3 order by device_id desc limit 100

Converting a Dataset to RDDs.



In [0]:
deviceEvents = df.select("device_name","cca3","c02_level").where(col("c02_level") > 1300)

eventsRDD = deviceEvents.take(10)


In [0]:
display(deviceEvents)

## ASSIGNMENT 3

##1. Explain the main differences between RDDs, Dataframes and
##Datasets (4 marks)

| Aspect | RDD | DataFrame | Dataset |
|---|---|---|---|
| What it is | Low-level, immutable, distributed collection of objects | Distributed data organised in named columns, like a table (Dataset[Row]) | Distributed collection of strongly typed objects (Dataset[T]) |
| Schema | None | Yes | Yes, defined by a case class |
| Type safety | Compile-time, but no structure | Column errors only show at runtime | Syntax and column errors caught at compile time |
| Optimisation | None (no Catalyst or Tungsten) | Catalyst and Tungsten | Catalyst and Tungsten |
| Languages | All | All, and the only option in Python and R | Scala and Java only |
| Best for | Unstructured data, fine-grained control | Structured data, SQL-style analysis | Structured data where type safety matters |

Query Setup

In [0]:
import pandas as pd

file_location = "https://raw.githubusercontent.com/dmatrix/examples/master/spark/databricks/notebooks/py/data/iot_devices.json"
pdf = pd.read_json(file_location, lines=True)
df_iot = spark.createDataFrame(pdf)
df_iot.createOrReplaceTempView("iot")

##2.1 How many sensor pads are reported to be from Poland (2 marks) 

In [0]:
%sql
SELECT
  COUNT(*) AS sensor_pads_poland
FROM
  iot
WHERE
  cn = 'Poland'
  AND device_name LIKE 'sensor-pad%';

In [0]:
import pyspark.sql.functions as F

poland_pads = df_iot.filter((F.col("cn") == "Poland") & F.col("device_name").startswith("sensor-pad")).count()
print("Sensor pads in Poland:", poland_pads)

2.1 Answer = 1,413 sensors pads

##2.2 How many different LCDs (distinct colors) are present in the
##dataset (2 marks)

In [0]:
%sql
SELECT COUNT(DISTINCT lcd) AS distinct_lcds FROM iot;
SELECT DISTINCT lcd FROM iot;

In [0]:
df_iot.select("lcd").distinct().show()
print("Distinct LCD colours:", df_iot.select("lcd").distinct().count())

2.2 Answer = 3 colors (green, yellow, red)

##2.3 Find 5 countries that have the largest number of MAC devices 
##used (2 marks)

In [0]:
%sql
SELECT cn, COUNT(*) AS mac_devices
FROM iot
WHERE device_name LIKE 'device-mac%'
GROUP BY cn
ORDER BY mac_devices DESC
LIMIT 5;

In [0]:
top_mac = (df_iot .filter(F.col("device_name").startswith("device-mac"))
             .groupBy("cn").count()
             .orderBy(F.desc("count"))
             .limit(5))
display(top_mac)

2.3 Answer = United States 11,508, China 2,300, Japan 2,001, Republic of Korea 1,999 and Germany 1,314.

##2.4 Propose and try an interesting statistical test or machine learning model.

You could use to gain insight from this dataset. Note, you don't have to use Machine Learning for this question. You can apply any analysis to the data even using SparkSQL, Python visualization libraries to analyze the data. Another example could be to apply correlation functions or other Spark functions to analyze the data. (2 marks)

## What drives the LCD status, and does device type matter?
1. Pearson correlation matrix of the numeric sensor readings (MLlib `Correlation`).


In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.stat import Correlation, ChiSquareTest

num_cols = ["temp", "humidity", "battery_level", "c02_level"]
vec = VectorAssembler(inputCols=num_cols, outputCol="features").transform(df_iot).select("features")
corr = Correlation.corr(vec, "features", "pearson").head()[0].toArray()

import pandas as pd
display(spark.createDataFrame(pd.DataFrame(corr, columns=num_cols).round(4).assign(variable=num_cols)))

**Analysis:** The sensor readings are independent of each other. The Pearson correlations between temperature, humidity, battery level and CO₂ are all close to zero (|r| < 0.004). No reading tells you anything about another.

In [0]:
#Code for vizualization using Python

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

PATH = "/FileStore/tables/iot_devices.json"  
df = spark.read.json(PATH)

LCD_COLOURS = {"green": "#2e9e44", "yellow": "#e6b800", "red": "#d62728"}
LCD_ORDER = ["green", "yellow", "red"]
num_cols = ["temp", "humidity", "battery_level", "c02_level"]

### Sensor readings are uncorrelated
All Pearson correlations between the readings are close to zero.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

vec = VectorAssembler(inputCols=num_cols, outputCol="features").transform(df_iot).select("features")
corr = Correlation.corr(vec, "features", "pearson").head()[0].toArray()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=30, ha="right")
ax.set_yticks(range(len(num_cols)))
ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr[i, j]:.3f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Correlation between sensor readings")
plt.tight_layout()
display(fig)

2. Chi-square test of independence between device type and LCD colour (MLlib `ChiSquareTest`).

In [0]:
# Chi-square: device type (prefix of device_name) vs LCD colour
typed = df_iot.withColumn("device_type", F.regexp_extract("device_name", r"^([a-z]+-[a-z]+)", 1))
display(typed.groupBy("device_type").pivot("lcd").count().orderBy("device_type"))

idx_type = StringIndexer(inputCol="device_type", outputCol="type_idx").fit(typed)
idx_lcd = StringIndexer(inputCol="lcd", outputCol="label").fit(typed)
chi_df = idx_lcd.transform(idx_type.transform(typed))
chi_df = VectorAssembler(inputCols=["type_idx"], outputCol="type_vec").transform(chi_df)

res = ChiSquareTest.test(chi_df, "type_vec", "label").head()
print("Chi-square statistic:", res.statistics, " dof:", res.degreesOfFreedom, " p-value:", res.pValues)

**Analysis:** Device type has no effect on the LCD colour. The chi-square test of device type against LCD colour gives χ² ≈ 8.91 with 6 degrees of freedom and p ≈ 0.18. That is above 0.05, so we can’t reject independence. Every device type shows about 25% green, 50% yellow and 25% red.

### Device type does not affect the LCD colour (chi-square p = 0.18)
Every device type shows the same split of roughly 25% green, 50% yellow and 25% red.

In [0]:
typed = df_iot.withColumn("device_type", F.regexp_extract("device_name", r"^([a-z]+-[a-z]+)", 1))
by_type = (typed.groupBy("device_type", "lcd").count()
                .toPandas()
                .pivot(index="device_type", columns="lcd", values="count")
                .reindex(columns=LCD_ORDER))
by_type_pct = by_type.div(by_type.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 4))
left = np.zeros(len(by_type_pct))
for lcd in LCD_ORDER:
    vals = by_type_pct[lcd].values
    ax.barh(by_type_pct.index, vals, left=left, color=LCD_COLOURS[lcd], label=lcd)
    for i, v in enumerate(vals):
        ax.text(left[i] + v / 2, i, f"{v:.1f}%", ha="center", va="center",
                color="black" if lcd == "yellow" else "white", fontsize=9)
    left += vals
ax.set_xlim(0, 100)
ax.set_xlabel("Share of devices (%)")
ax.set_title("LCD colour share by device type")
ax.legend(title="LCD", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
display(fig)

3. Decision tree classifier predicting LCD colour from the readings, to discover the rule behind it.

In [0]:
# Decision tree: predict LCD colour from sensor readings
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

train, test = df_iot.randomSplit([0.7, 0.3], seed=42)
pipe = Pipeline(stages=[
    StringIndexer(inputCol="lcd", outputCol="label"),
    VectorAssembler(inputCols=num_cols, outputCol="features"),
    DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=3),
])
model = pipe.fit(train)
pred = model.transform(test)
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
print("Test accuracy:", acc)

tree = model.stages[-1]
print("Label order:", model.stages[0].labels)
print("Feature importances:", dict(zip(num_cols, tree.featureImportances.toArray())))
print(tree.toDebugString)

In [0]:
# Confirm the thresholds the tree found
display(df_iot.groupBy("lcd").agg(F.min("c02_level").alias("min_co2"),
                              F.max("c02_level").alias("max_co2"),
                              F.count("*").alias("devices")).orderBy("min_co2"))

**Analysis:** CO₂ alone determines the LCD colour. The decision tree predicts the colour with 100% test accuracy, CO₂ has a feature importance of 1.0, and the other readings have none. Without CO₂, accuracy drops to about 50%, which is no better than always guessing “yellow”. The tree recovers the exact rule, and the final query in the notebook confirms it:
green when CO₂ is 800–1000
yellow when CO₂ is 1001–1400
red when CO₂ is 1401–1599

### CO2 is the only feature the decision tree uses
The tree reaches 100% test accuracy, with all of its feature importance on CO2.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

train, test = df_iot.randomSplit([0.7, 0.3], seed=42)
model = Pipeline(stages=[
    StringIndexer(inputCol="lcd", outputCol="label"),
    VectorAssembler(inputCols=num_cols, outputCol="features"),
    DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=3),
]).fit(train)
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(model.transform(test))
importances = pd.Series(model.stages[-1].featureImportances.toArray(), index=num_cols).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(importances.index, importances.values,
        color=["#d62728" if c == "c02_level" else "#999999" for c in importances.index])
for i, v in enumerate(importances.values):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
ax.set_xlim(0, 1.15)
ax.set_xlabel("Feature importance")
ax.set_title(f"Decision tree feature importance (test accuracy = {acc:.1%})")
plt.tight_layout()
display(fig)

### CO2 level determines the LCD colour
Each colour occupies its own CO2 band with no overlap, so the LCD is a CO2 alarm.

In [0]:
# Bins run 801-810, 811-820, ... so that 1000 and 1400 fall on bin edges (800 joins the first bin)
co2_bins = (df_iot.withColumn("co2_bin", (F.greatest(F.floor((F.col("c02_level") - 1) / 10), F.lit(80)) * 10 + 1).cast("int"))
              .groupBy("co2_bin", "lcd").count()
              .toPandas()
              .pivot(index="co2_bin", columns="lcd", values="count")
              .fillna(0)
              .reindex(columns=LCD_ORDER, fill_value=0)
              .sort_index())

fig, ax = plt.subplots(figsize=(11, 4.5))
bottom = np.zeros(len(co2_bins))
for lcd in LCD_ORDER:
    ax.bar(co2_bins.index - 0.5, co2_bins[lcd], width=10, bottom=bottom, align="edge",
           color=LCD_COLOURS[lcd], label=lcd)
    bottom += co2_bins[lcd].values
for x in (1000.5, 1400.5):
    ax.axvline(x, color="black", linestyle="--", linewidth=1)
ax.text(900, bottom.max() * 1.02, "green\n<= 1000", ha="center")
ax.text(1200, bottom.max() * 1.02, "yellow\n1001-1400", ha="center")
ax.text(1500, bottom.max() * 1.02, "red\n>= 1401", ha="center")
ax.set_ylim(0, bottom.max() * 1.18)
ax.set_xlabel("CO2 level (ppm, 10-ppm bins)")
ax.set_ylabel("Number of devices")
ax.set_title("Distribution of CO2 level by LCD colour")
ax.legend(title="LCD", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
display(fig)

###Analysis with SQL queries

In [0]:
%sql
-- Correlation between sensor readings (all close to 0)
SELECT corr(temp, humidity)        AS temp_humidity,
       corr(temp, c02_level)       AS temp_co2,
       corr(humidity, c02_level)   AS humidity_co2,
       corr(battery_level, temp)   AS battery_temp
FROM iot;

In [0]:
%sql
-- LCD colour vs CO2 level
SELECT lcd, MIN(c02_level) AS min_co2, MAX(c02_level) AS max_co2, COUNT(*) AS devices
FROM iot
GROUP BY lcd
ORDER BY min_co2;

**Analysis:** The data appears to be synthetic. The near-zero correlations and the even spread of colours across device types suggest the readings were generated at random rather than measured.

In short, the LCD is a CO₂ air-quality alarm. You can monitor air quality directly from the lcd field, and temperature, humidity, battery level and device type tell you nothing about it.
